# HidraImg Worker no Google Colab

Notebook pronto para instalar dependencias, baixar modelo Stable Diffusion e iniciar o worker HidraChat com GPU CUDA.

Antes de rodar: em `Runtime > Change runtime type`, selecione uma GPU.

In [ ]:
# 1. Conferir GPU
!nvidia-smi

In [ ]:
# 2. Baixar o worker do GitHub
%cd /content
!rm -rf hidrachat-image-worker
!git clone https://github.com/Luispessoa18/hidrachat-image-worker
%cd /content/hidrachat-image-worker

In [ ]:
# 3. Instalar PyTorch CUDA + Diffusers
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q xformers

## Configuracao

O email padrao ja esta definido como `luispessoa18@gmail.com`.

O modelo padrao abaixo e SD 1.5, mais leve para testar em T4. Para SDXL, troque `MODEL_ID` por `stabilityai/stable-diffusion-xl-base-1.0`.

In [ ]:
# 4. Definir email e modelo
import os

EMAIL = "luispessoa18@gmail.com"
MODEL_ID = "runwayml/stable-diffusion-v1-5"
LOCAL_MODEL_DIR = "/content/hidrachat-image-worker/models/sd15"

os.environ["HIDRACHAT_WORKER_EMAIL"] = EMAIL
os.environ["HIDRACHAT_MODEL_ID"] = LOCAL_MODEL_DIR
os.environ["HIDRACHAT_DEVICE"] = "cuda"
os.environ["HIDRACHAT_TORCH_DTYPE"] = "auto"
os.environ["HIDRACHAT_WORKER_NAME"] = "image-worker-colab"
os.environ["HIDRACHAT_REGION"] = "colab"
os.environ["HIDRACHAT_LOCAL_FILES_ONLY"] = "1"
os.environ["HIDRACHAT_PRELOAD_MODEL"] = "1"

print("Email:", os.environ["HIDRACHAT_WORKER_EMAIL"])
print("Modelo local:", os.environ["HIDRACHAT_MODEL_ID"])

## Baixar modelo

Se o Hugging Face pedir aceite de licenca/token, rode a celula de login primeiro e cole seu token.

In [ ]:
# Opcional: login no Hugging Face, caso o modelo exija token
# from huggingface_hub import login
# login()

In [ ]:
# 5. Baixar somente os arquivos necessarios do modelo
!python download_model.py --model {MODEL_ID} --output {LOCAL_MODEL_DIR}

In [ ]:
# 6. Iniciar worker depois do modelo local estar pronto
!python colab_worker.py --model {LOCAL_MODEL_DIR}

## Alternativa online, sem baixar modelo antes

Se quiser deixar o Diffusers baixar/cachear automaticamente durante o start, rode isto no lugar das celulas 4 e 5:

```python
%env HIDRACHAT_WORKER_EMAIL=luispessoa18@gmail.com
%env HIDRACHAT_MODEL_ID=runwayml/stable-diffusion-v1-5
%env HIDRACHAT_DEVICE=cuda
!python colab_worker.py --online-model
```